# YatraAI: Maharashtra Monuments YOLO11n Training Notebook
### AI/ML Computer Vision Pipeline for 29 Maharashtra Heritage Landmarks

This Google Colab notebook provides the complete workflow to train an **Ultralytics YOLO11n** object detection model on free T4/A100 GPU hardware, evaluate detection accuracy, and export `best.pt` to power the **YatraAI** travel companion application.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

## 2. Install Ultralytics & Computer Vision Dependencies

In [ ]:
!pip install --upgrade ultralytics opencv-python matplotlib pillow

## 3. Upload and Extract Dataset (`YatraAI_Maharashtra_Dataset.zip`)
Upload your partitioned dataset zip containing `train/`, `valid/`, `test/`, and `data.yaml`.

In [ ]:
# If using Google Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')

# Or unzip uploaded dataset archive:
!unzip -q YatraAI_Maharashtra_Dataset.zip -d ./dataset
!ls -lh ./dataset

## 4. Verify `data.yaml` Configuration (29 Classes)

In [ ]:
yaml_content = """
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 29
names:
  0: Raigad Fort
  1: Shivneri Fort
  2: Sinhagad Fort
  3: Pratapgad Fort
  4: Aga Khan Palace
  5: Gateway of India
  6: Ajanta Caves
  7: Ellora Caves
  8: Daulatabad Fort
  9: Shaniwar Wada
  10: Mahalaxmi Temple
  11: Dagdusheth Halwai Ganpati Temple
  12: Marine Drive
  13: Elephanta Caves
  14: Thoseghar Waterfall
  15: Rankala Lake
  16: Ganpatipule Beach
  17: Tarkarli Beach
  18: Trimbakeshwar Temple
  19: Panhala Fort
  20: Sindhudurg Fort
  21: Vijaydurg Fort
  22: Lohagad Fort
  23: Torna Fort
  24: Bibi Ka Maqbara
  25: Bhimashankar Temple
  26: Pandharpur Vitthal Temple
  27: Grishneshwar Temple
  28: Kanheri Caves
"""
with open('/content/dataset/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

!cat /content/dataset/data.yaml

## 5. Train Ultralytics YOLO11n Model (100 Epochs, Image Size 640)

In [ ]:
from ultralytics import YOLO

# Load lightweight base model
model = YOLO('yolo11n.pt')

# Start training
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='runs/train',
    name='yatraai_yolo11n',
    patience=20,
    cos_lr=True,
    save=True,
    plots=True,
    verbose=True
)

## 6. Visualize Training Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display

# Display loss and mAP curves
display(Image(filename='runs/train/yatraai_yolo11n/results.png'))

# Display confusion matrix
display(Image(filename='runs/train/yatraai_yolo11n/confusion_matrix.png'))

## 7. Model Evaluation on Unseen Test Split

In [ ]:
# Load best checkpoint and validate on test split
best_model = YOLO('runs/train/yatraai_yolo11n/weights/best.pt')
test_metrics = best_model.val(data='/content/dataset/data.yaml', split='test', imgsz=640)

print("=" * 60)
print(f"Test Precision (P)   : {test_metrics.box.mp * 100:.2f}%")
print(f"Test Recall (R)      : {test_metrics.box.mr * 100:.2f}%")
print(f"Test mAP @ 0.50      : {test_metrics.box.map50 * 100:.2f}%")
print(f"Test mAP @ 0.50:0.95 : {test_metrics.box.map * 100:.2f}%")
print("=" * 60)

## 8. Run Visual Inference Test on Sample Landmark Photo

In [ ]:
# Run inference on a test photo
preds = best_model.predict(source='/content/dataset/test/images', save=True, conf=0.40, max_det=3)

# Display first sample result
import glob
sample_preds = glob.glob('runs/detect/predict*/*.jpg')
if sample_preds:
    display(Image(filename=sample_preds[0]))

## 9. Download `best.pt` for YatraAI Application Deployment

In [ ]:
from google.colab import files
# Download the best weights to place into backend/models/best.pt
files.download('runs/train/yatraai_yolo11n/weights/best.pt')